<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:15px; color:white; margin:0; font-size:150%; font-family:Pacifico; background-color:#8B0000; overflow:hidden"><b> Predicting Student Test Scores </b></div>

## 📌 Notebook Introduction

In this notebook, I build an end-to-end regression pipeline to predict student test scores using demographic, academic, and behavioral features.
The solution focuses on robust cross-validation, powerful gradient boosting models (CatBoost and LightGBM), and out-of-fold (OOF) evaluation to ensure reliable performance estimation.

Key aspects of this approach include:

Careful preprocessing of numerical and categorical features

K-Fold cross-validation with OOF predictions

Model comparison and ensembling to improve generalization

RMSE as the primary evaluation metric

### The goal is not only to achieve a strong leaderboard score, but also to maintain a clean, reproducible, and well-validated modeling strategy suitable for real-world regression problems.

In [8]:
!pip install lightgbm catboost

<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#3168a1; overflow:hidden"><b> Imports </b></div>

In [9]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

from catboost import CatBoostRegressor
import lightgbm as lgb

import warnings
warnings.filterwarnings("ignore")

SEED = 42
N_SPLITS = 7


In [10]:
train = pd.read_csv("/kaggle/input/playground-series-s6e1/train.csv")
test  = pd.read_csv("/kaggle/input/playground-series-s6e1/test.csv")

print(train.shape, test.shape)


(630000, 13) (270000, 12)


<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#3168a1; overflow:hidden"><b> feature_engineering </b></div>

In [11]:
def feature_engineering(df):
    df = df.copy()
    
    df["study_sleep_ratio"] = df["study_hours"] / (df["sleep_hours"] + 1)
    df["attendance_study_ratio"] = df["class_attendance"] / (df["study_hours"] + 1)
    df["study_sleep_diff"] = df["study_hours"] - df["sleep_hours"]
    df["attendance_sleep_ratio"] = df["class_attendance"] / (df["sleep_hours"] + 1)
    df["study_squared"] = df["study_hours"] ** 2
    df["sleep_squared"] = df["sleep_hours"] ** 2

    df["sleep_score"] = df["sleep_hours"] * df["sleep_quality"].map({
        "poor": 1, "average": 2, "good": 3
    })
    
    return df

train = feature_engineering(train)
test  = feature_engineering(test)


In [12]:
TARGET = "exam_score"

X = train.drop(columns=[TARGET, "id"])
y = train[TARGET]

test_X = test.drop(columns=["id"])

cat_features = X.select_dtypes(include="object").columns.tolist()


In [13]:
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

oof_cat = np.zeros(len(X))
test_cat = np.zeros(len(test_X))

oof_lgb = np.zeros(len(X))
test_lgb = np.zeros(len(test_X))


<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#3168a1; overflow:hidden"><b> CatBoost Model</b></div>

In [14]:
for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    print(f"\n CatBoost Fold {fold+1}")

    X_train, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = CatBoostRegressor(
    iterations=4000,
    learning_rate=0.03,
    depth=6,
    loss_function="RMSE",
    eval_metric="RMSE",
    task_type="GPU",
    devices="0",
    bootstrap_type="Bayesian",
    cat_features=cat_features,   
    verbose=300
)


    model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        cat_features=cat_features,
        early_stopping_rounds=200      
    )

    oof_cat[val_idx] = model.predict(X_val)
    test_cat += model.predict(test_X) / N_SPLITS



 CatBoost Fold 1
0:	learn: 18.5507017	test: 18.4720486	best: 18.4720486 (0)	total: 55.6ms	remaining: 3m 42s
300:	learn: 8.8378093	test: 8.7985722	best: 8.7985722 (300)	total: 12.1s	remaining: 2m 28s
600:	learn: 8.8142276	test: 8.7806334	best: 8.7806334 (600)	total: 24s	remaining: 2m 15s
900:	learn: 8.7980507	test: 8.7709318	best: 8.7709318 (900)	total: 36s	remaining: 2m 3s
1200:	learn: 8.7854585	test: 8.7648363	best: 8.7648363 (1200)	total: 48s	remaining: 1m 51s
1500:	learn: 8.7742840	test: 8.7599845	best: 8.7599845 (1500)	total: 1m	remaining: 1m 40s
1800:	learn: 8.7643520	test: 8.7563348	best: 8.7563348 (1800)	total: 1m 12s	remaining: 1m 28s
2100:	learn: 8.7550629	test: 8.7536237	best: 8.7535901 (2096)	total: 1m 24s	remaining: 1m 16s
2400:	learn: 8.7465075	test: 8.7512805	best: 8.7512805 (2400)	total: 1m 36s	remaining: 1m 4s
2700:	learn: 8.7384167	test: 8.7493882	best: 8.7493812 (2686)	total: 1m 49s	remaining: 52.6s
3000:	learn: 8.7304889	test: 8.7475409	best: 8.7475409 (3000)	total:

<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#3168a1; overflow:hidden"><b> LightGBM Model</b></div>

In [15]:
from sklearn.preprocessing import LabelEncoder

X_lgb = X.copy()
test_X_lgb = test_X.copy()

for col in cat_features:
    le = LabelEncoder()
    X_lgb[col] = le.fit_transform(X_lgb[col].astype(str))
    test_X_lgb[col] = le.transform(test_X_lgb[col].astype(str))


In [16]:
for fold, (train_idx, val_idx) in enumerate(kf.split(X_lgb)):
    print(f"\n LightGBM Fold {fold+1}")
    
    X_train, X_val = X_lgb.iloc[train_idx], X_lgb.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    lgb_model = lgb.LGBMRegressor(
        n_estimators=6000,
        learning_rate=0.03,
        max_depth=-1,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=SEED
    )

    lgb_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric="rmse",
        callbacks=[lgb.early_stopping(300)]
    )

    oof_lgb[val_idx] = lgb_model.predict(X_val)
    test_lgb += lgb_model.predict(test_X_lgb) / N_SPLITS



 LightGBM Fold 1
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.052984 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2085
[LightGBM] [Info] Number of data points in the train set: 540000, number of used features: 18
[LightGBM] [Info] Start training from score 62.494868
Training until validation scores don't improve for 300 rounds
Early stopping, best iteration is:
[3202]	valid_0's rmse: 8.71983	valid_0's l2: 76.0354

 LightGBM Fold 2
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.054648 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2084
[LightGBM] [Info] Number of data points in the train set: 540000, number of used features: 18
[LightGBM] [Info] Start training from score 62.503715
Training until validation scores don't improve for 300 rounds
Early stopping, best iteration is:
[2895]	valid_0's rmse: 8.7658

In [17]:
!pip install --upgrade scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 63.1 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fastai 2.8.4 requires fastcore<1.9,>=1.8.0, but you have fastcore 1.11.3 which is incompatible.


In [18]:
cat_rmse = mean_squared_error(y, oof_cat)
lgb_rmse = mean_squared_error(y, oof_lgb)

print("CatBoost CV RMSE:",np.sqrt(cat_rmse) )
print("LightGBM CV RMSE:", np.sqrt(lgb_rmse))


CatBoost CV RMSE: 8.785616251950259
LightGBM CV RMSE: 8.760378720035689


In [20]:

stack_X = np.vstack([oof_cat, oof_lgb]).T
weights, _, _, _ = np.linalg.lstsq(stack_X, y, rcond=None)

print("Weights:", weights)

oof_final = stack_X @ weights
test_final = np.vstack([test_cat, test_lgb]).T @ weights


Weights: [0.21066392 0.78942187]


In [21]:
submission = pd.DataFrame({
    "id": test["id"],
    "exam_score": test_final
})

submission.to_csv("submission.csv", index=False)
submission.head()


,id,exam_score
0,630000,72.088572
1,630001,70.329908
2,630002,88.594597
3,630003,56.011713
4,630004,47.301058
